In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
try:
    df = pd.read_csv('spam.csv', encoding='latin-1')

    # Keep only the necessary columns (usually v1 for label, v2 for text)
    df = df[['v1', 'v2']]
    df.columns = ['label', 'text']

    # Map labels to binary
    df['label'] = df['label'].map({'ham': 0, 'spam': 1})

    X = df['text']
    y = df['label']

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # --- Model 1: CountVectorizer ---
    cv = CountVectorizer(stop_words='english')
    X_train_cv = cv.fit_transform(X_train)
    X_test_cv = cv.transform(X_test)

    nb_cv = MultinomialNB()
    nb_cv.fit(X_train_cv, y_train)
    y_pred_cv = nb_cv.predict(X_test_cv)

    acc_cv = accuracy_score(y_test, y_pred_cv)
    rep_cv = classification_report(y_test, y_pred_cv, target_names=['ham', 'spam'], output_dict=True)

    # --- Model 2: TF-IDF Vectorizer ---
    tfidf = TfidfVectorizer(stop_words='english')
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    nb_tfidf = MultinomialNB()
    nb_tfidf.fit(X_train_tfidf, y_train)
    y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

    acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
    rep_tfidf = classification_report(y_test, y_pred_tfidf, target_names=['ham', 'spam'], output_dict=True)

    # Output the results for comparison
    print(f"--- Evaluasi CountVectorizer ---")
    print(f"Akurasi: {acc_cv:.4f}")
    print(f"Precision (Spam): {rep_cv['spam']['precision']:.4f}")
    print(f"Recall (Spam): {rep_cv['spam']['recall']:.4f}")
    print(f"F1-Score (Spam): {rep_cv['spam']['f1-score']:.4f}")
    print("\n--- Evaluasi TF-IDF ---")
    print(f"Akurasi: {acc_tfidf:.4f}")
    print(f"Precision (Spam): {rep_tfidf['spam']['precision']:.4f}")
    print(f"Recall (Spam): {rep_tfidf['spam']['recall']:.4f}")
    print(f"F1-Score (Spam): {rep_tfidf['spam']['f1-score']:.4f}")

    # Plot confusion matrices for a nicer output
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.heatmap(confusion_matrix(y_test, y_pred_cv), annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
    axes[0].set_title('Confusion Matrix - CountVectorizer')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')

    sns.heatmap(confusion_matrix(y_test, y_pred_tfidf), annot=True, fmt='d', cmap='Greens', ax=axes[1], xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
    axes[1].set_title('Confusion Matrix - TF-IDF')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig('spam_comparison.png')
    plt.close()

except Exception as e:
    print(f"Error: {e}")

--- Evaluasi CountVectorizer ---
Akurasi: 0.9839
Precision (Spam): 0.9583
Recall (Spam): 0.9200
F1-Score (Spam): 0.9388

--- Evaluasi TF-IDF ---
Akurasi: 0.9668
Precision (Spam): 1.0000
Recall (Spam): 0.7533
F1-Score (Spam): 0.8593

Visualizations saved successfully.


Untuk kasus data spam.csv menggunakan model Multinomial Naive Bayes, CountVectorizer adalah fitur yang terbaik.

Alasannya: Dalam kasus deteksi SMS spam dengan algoritma Naive Bayes, data teks umumnya pendek-pendek (seperti SMS pada umumnya). Metode TF-IDF justru melemahkan bobot kata-kata yang sering muncul namun mungkin merupakan indikator kuat dari pesan spam (misalnya "FREE", "Win", "Click"). Akibatnya, TF-IDF menyebabkan model gagal mengenali banyak pesan spam (Recall rendah). Di sisi lain, CountVectorizer hanya mengandalkan perhitungan frekuensi mentah, yang terbukti lebih sensitif dan efektif dalam mengenali pola kata-kata spam pada pesan-pesan singkat tersebut sehingga menghasilkan F1-Score dan Akurasi keseluruhan yang lebih tinggi.